In [ ]:
## 학습 데이터 생성 코드구간 ##

In [1]:
# 1번 이미지 학습데이터 생성

import cv2
import numpy as np
import os
import random
import math

# 경로 설정
input_path = "/home/zen35/Desktop/genimage/1.png"
output_dir = "/home/zen35/Desktop/genimage/gen1"
os.makedirs(output_dir, exist_ok=True)

# 이미지 불러오기
img = cv2.imread(input_path)
h, w = img.shape[:2]

# 재단선 내부 좌표 (x, y, width, height)
inner_rect = (188, 283, 2103, 2942)
x, y, w_rect, h_rect = inner_rect

# 랜덤 배경 생성 함수
def generate_random_background(width, height):
    #bg_color = random.choice([(240, 240, 240), (200, 200, 200), (100, 100, 100)])

    bg_color = (random.randint(100, 255),random.randint(100, 255),random.randint(100, 255))
    
    background = np.full((height, width, 3), bg_color, dtype=np.uint8)
    noise = np.random.randint(0, 30, (height, width, 3), dtype=np.uint8)
    background = cv2.add(background, noise)
    return background

# 랜덤 도형 그리기 함수
def draw_random_shapes(image, count=20, region=None):
    height, width = image.shape[:2]

    def draw_star(center, size, color):
        pts = []
        for i in range(10):
            angle = i * math.pi / 5
            r = size if i % 2 == 0 else size / 2
            x = int(center[0] + r * math.cos(angle))
            y = int(center[1] + r * math.sin(angle))
            pts.append((x, y))
        pts = np.array(pts, np.int32)
        cv2.fillPoly(image, [pts], color)

    def draw_rotated_rect(center, size, angle_deg, color):
        rect = ((center[0], center[1]), (size[0], size[1]), angle_deg)
        box = cv2.boxPoints(rect).astype(np.int32)
        cv2.fillPoly(image, [box], color)

    def draw_rounded_triangle(center, size, color):
        x, y = center
        pts = np.array([
            [x, y - size],
            [x - size, y + size],
            [x + size, y + size]
        ], np.int32)
        cv2.fillPoly(image, [pts], color)

    for _ in range(count):
        shape_type = random.choice([
            "circle", "rectangle", "line", "triangle",
            "rotated_rectangle", "star", "rounded_triangle"
        ])
        
        
        #color = random.choice([(180, 180, 180), (80, 40, 40)])
        color = (random.randint(50, 255),random.randint(50, 255),random.randint(50, 255))

        
        thickness = random.randint(1, 3)

        if region:
            rx, ry, rw, rh = region
            cx = random.randint(rx, rx + rw)
            cy = random.randint(ry, ry + rh)
        else:
            cx, cy = random.randint(0, width), random.randint(0, height)

        if shape_type == "circle":
            radius = random.randint(10, 40)
            cv2.circle(image, (cx, cy), radius, color, -1)

        elif shape_type == "rectangle":
            w_, h_ = random.randint(30, 80), random.randint(30, 80)
            cv2.rectangle(image, (cx, cy), (cx + w_, cy + h_), color, -1)

        elif shape_type == "line":
            pt2 = (random.randint(0, width), random.randint(0, height))
            cv2.line(image, (cx, cy), pt2, color, thickness)

        elif shape_type == "triangle":
            pts = np.array([
                [cx, cy],
                [cx + random.randint(20, 50), cy + random.randint(20, 50)],
                [cx - random.randint(20, 50), cy + random.randint(20, 50)]
            ], np.int32)
            cv2.fillPoly(image, [pts], color)

        elif shape_type == "rotated_rectangle":
            draw_rotated_rect((cx, cy), (random.randint(40, 80), random.randint(40, 80)), random.randint(10, 75), color)

        elif shape_type == "star":
            draw_star((cx, cy), random.randint(20, 40), color)

        elif shape_type == "rounded_triangle":
            draw_rounded_triangle((cx, cy), random.randint(25, 40), color)

    return image

# 이미지 여러 장 생성
num_images = 200
for i in range(num_images):
    img_copy = img.copy()

    # 기본 배경 + 랜덤 도형 전체 배치
    rand_bg = generate_random_background(w_rect, h_rect)
    rand_bg = draw_random_shapes(rand_bg, count=200)

    # 책등/가장자리에 도형 집중 배치할 영역 정의
    margin = 200
    region_defs = [
        # 4 모서리
        (0, 0, margin, margin),  # 좌상단
        (w_rect - margin, 0, margin, margin),  # 우상단
        (0, h_rect - margin, margin, margin),  # 좌하단
        (w_rect - margin, h_rect - margin, margin, margin),  # 우하단

        # 위/아래 중앙
        (w_rect // 2 - margin // 2, 0, margin, margin),  # 상단 중앙
        (w_rect // 2 - margin // 2, h_rect - margin, margin, margin),  # 하단 중앙

        # 좌/우 세로 중앙 (책등 강조)
        (0, h_rect // 2 - margin // 2, margin, margin),  # 왼쪽 세로 중앙
        (w_rect - margin, h_rect // 2 - margin // 2, margin, margin),  # 오른쪽 세로 중앙
    ]

    # 각 영역에 도형 n개씩 추가
    for region in region_defs:
        rand_bg = draw_random_shapes(rand_bg, count=15, region=region)

    # 재단선 영역에 최종 이미지 삽입
    img_copy[y:y + h_rect, x:x + w_rect] = rand_bg

    # 저장
    save_path = os.path.join(output_dir, f"1_{i}.png")
    cv2.imwrite(save_path, img_copy)

print(f"{num_images}개의 이미지가 생성되었습니다")


200개의 이미지가 생성되었습니다.


In [2]:
# 2번 이미지 학습데이터 생성

import cv2
import numpy as np
import os
import random
import math

# 경로 설정
input_path = "/home/zen35/Desktop/genimage/2.png"
output_dir = "/home/zen35/Desktop/genimage/gen2"
os.makedirs(output_dir, exist_ok=True)

# 이미지 불러오기
img = cv2.imread(input_path)
h, w = img.shape[:2]

# 재단선 내부 좌표 (x, y, width, height)
inner_rect = (70, 70, 5295, 3345)
x, y, w_rect, h_rect = inner_rect

# 랜덤 배경 생성 함수
def generate_random_background(width, height):
    #bg_color = random.choice([(240, 240, 240), (200, 200, 200), (100, 100, 100)])

    bg_color = (random.randint(100, 255),random.randint(100, 255),random.randint(100, 255))
    
    background = np.full((height, width, 3), bg_color, dtype=np.uint8)
    noise = np.random.randint(0, 30, (height, width, 3), dtype=np.uint8)
    background = cv2.add(background, noise)
    return background

# 랜덤 도형 그리기 함수
def draw_random_shapes(image, count=20, region=None):
    height, width = image.shape[:2]

    def draw_star(center, size, color):
        pts = []
        for i in range(10):
            angle = i * math.pi / 5
            r = size if i % 2 == 0 else size / 2
            x = int(center[0] + r * math.cos(angle))
            y = int(center[1] + r * math.sin(angle))
            pts.append((x, y))
        pts = np.array(pts, np.int32)
        cv2.fillPoly(image, [pts], color)

    def draw_rotated_rect(center, size, angle_deg, color):
        rect = ((center[0], center[1]), (size[0], size[1]), angle_deg)
        box = cv2.boxPoints(rect).astype(np.int32)
        cv2.fillPoly(image, [box], color)

    def draw_rounded_triangle(center, size, color):
        x, y = center
        pts = np.array([
            [x, y - size],
            [x - size, y + size],
            [x + size, y + size]
        ], np.int32)
        cv2.fillPoly(image, [pts], color)

    for _ in range(count):
        shape_type = random.choice([
            "circle", "rectangle", "line", "triangle",
            "rotated_rectangle", "star", "rounded_triangle"
        ])
        
        
        #color = random.choice([(180, 180, 180), (80, 40, 40)])
        color = (random.randint(50, 255),random.randint(50, 255),random.randint(50, 255))

        
        thickness = random.randint(1, 3)

        if region:
            rx, ry, rw, rh = region
            cx = random.randint(rx, rx + rw)
            cy = random.randint(ry, ry + rh)
        else:
            cx, cy = random.randint(0, width), random.randint(0, height)

        if shape_type == "circle":
            radius = random.randint(10, 40)
            cv2.circle(image, (cx, cy), radius, color, -1)

        elif shape_type == "rectangle":
            w_, h_ = random.randint(30, 80), random.randint(30, 80)
            cv2.rectangle(image, (cx, cy), (cx + w_, cy + h_), color, -1)

        elif shape_type == "line":
            pt2 = (random.randint(0, width), random.randint(0, height))
            cv2.line(image, (cx, cy), pt2, color, thickness)

        elif shape_type == "triangle":
            pts = np.array([
                [cx, cy],
                [cx + random.randint(20, 50), cy + random.randint(20, 50)],
                [cx - random.randint(20, 50), cy + random.randint(20, 50)]
            ], np.int32)
            cv2.fillPoly(image, [pts], color)

        elif shape_type == "rotated_rectangle":
            draw_rotated_rect((cx, cy), (random.randint(40, 80), random.randint(40, 80)), random.randint(10, 75), color)

        elif shape_type == "star":
            draw_star((cx, cy), random.randint(20, 40), color)

        elif shape_type == "rounded_triangle":
            draw_rounded_triangle((cx, cy), random.randint(25, 40), color)

    return image

# 이미지 여러 장 생성
num_images = 200
for i in range(num_images):
    img_copy = img.copy()

    # 기본 배경 + 랜덤 도형 전체 배치
    rand_bg = generate_random_background(w_rect, h_rect)
    rand_bg = draw_random_shapes(rand_bg, count=200)

    # 책등/가장자리에 도형 집중 배치할 영역 정의
    margin = 200
    region_defs = [
        # 4 모서리
        (0, 0, margin, margin),  # 좌상단
        (w_rect - margin, 0, margin, margin),  # 우상단
        (0, h_rect - margin, margin, margin),  # 좌하단
        (w_rect - margin, h_rect - margin, margin, margin),  # 우하단

        # 위/아래 중앙
        (w_rect // 2 - margin // 2, 0, margin, margin),  # 상단 중앙
        (w_rect // 2 - margin // 2, h_rect - margin, margin, margin),  # 하단 중앙

        # 좌/우 세로 중앙 (책등 강조)
        (0, h_rect // 2 - margin // 2, margin, margin),  # 왼쪽 세로 중앙
        (w_rect - margin, h_rect // 2 - margin // 2, margin, margin),  # 오른쪽 세로 중앙
    ]

    # 각 영역에 도형 n개씩 추가
    for region in region_defs:
        rand_bg = draw_random_shapes(rand_bg, count=15, region=region)

    # 재단선 영역에 최종 이미지 삽입
    img_copy[y:y + h_rect, x:x + w_rect] = rand_bg

    # 저장
    save_path = os.path.join(output_dir, f"2_{i}.png")
    cv2.imwrite(save_path, img_copy)

print(f"{num_images}개의 이미지가 생성되었습니다")


200개의 이미지가 생성되었습니다.


In [3]:
# 3번 이미지 학습데이터 생성

import cv2
import numpy as np
import os
import random
import math

# 경로 설정
input_path = "/home/zen35/Desktop/genimage/3.png"
output_dir = "/home/zen35/Desktop/genimage/gen3"
os.makedirs(output_dir, exist_ok=True)

# 이미지 불러오기
img = cv2.imread(input_path)
h, w = img.shape[:2]

# 재단선 내부 좌표 (x, y, width, height)
inner_rect = (122, 130, 5075, 3550)
x, y, w_rect, h_rect = inner_rect

# 랜덤 배경 생성 함수
def generate_random_background(width, height):
    #bg_color = random.choice([(240, 240, 240), (200, 200, 200), (100, 100, 100)])

    bg_color = (random.randint(100, 255),random.randint(100, 255),random.randint(100, 255))
    
    background = np.full((height, width, 3), bg_color, dtype=np.uint8)
    noise = np.random.randint(0, 30, (height, width, 3), dtype=np.uint8)
    background = cv2.add(background, noise)
    return background

# 랜덤 도형 그리기 함수
def draw_random_shapes(image, count=20, region=None):
    height, width = image.shape[:2]

    def draw_star(center, size, color):
        pts = []
        for i in range(10):
            angle = i * math.pi / 5
            r = size if i % 2 == 0 else size / 2
            x = int(center[0] + r * math.cos(angle))
            y = int(center[1] + r * math.sin(angle))
            pts.append((x, y))
        pts = np.array(pts, np.int32)
        cv2.fillPoly(image, [pts], color)

    def draw_rotated_rect(center, size, angle_deg, color):
        rect = ((center[0], center[1]), (size[0], size[1]), angle_deg)
        box = cv2.boxPoints(rect).astype(np.int32)
        cv2.fillPoly(image, [box], color)

    def draw_rounded_triangle(center, size, color):
        x, y = center
        pts = np.array([
            [x, y - size],
            [x - size, y + size],
            [x + size, y + size]
        ], np.int32)
        cv2.fillPoly(image, [pts], color)

    for _ in range(count):
        shape_type = random.choice([
            "circle", "rectangle", "line", "triangle",
            "rotated_rectangle", "star", "rounded_triangle"
        ])
        
        
        #color = random.choice([(180, 180, 180), (80, 40, 40)])
        color = (random.randint(50, 255),random.randint(50, 255),random.randint(50, 255))

        
        thickness = random.randint(1, 3)

        if region:
            rx, ry, rw, rh = region
            cx = random.randint(rx, rx + rw)
            cy = random.randint(ry, ry + rh)
        else:
            cx, cy = random.randint(0, width), random.randint(0, height)

        if shape_type == "circle":
            radius = random.randint(10, 40)
            cv2.circle(image, (cx, cy), radius, color, -1)

        elif shape_type == "rectangle":
            w_, h_ = random.randint(30, 80), random.randint(30, 80)
            cv2.rectangle(image, (cx, cy), (cx + w_, cy + h_), color, -1)

        elif shape_type == "line":
            pt2 = (random.randint(0, width), random.randint(0, height))
            cv2.line(image, (cx, cy), pt2, color, thickness)

        elif shape_type == "triangle":
            pts = np.array([
                [cx, cy],
                [cx + random.randint(20, 50), cy + random.randint(20, 50)],
                [cx - random.randint(20, 50), cy + random.randint(20, 50)]
            ], np.int32)
            cv2.fillPoly(image, [pts], color)

        elif shape_type == "rotated_rectangle":
            draw_rotated_rect((cx, cy), (random.randint(40, 80), random.randint(40, 80)), random.randint(10, 75), color)

        elif shape_type == "star":
            draw_star((cx, cy), random.randint(20, 40), color)

        elif shape_type == "rounded_triangle":
            draw_rounded_triangle((cx, cy), random.randint(25, 40), color)

    return image

# 이미지 여러 장 생성
num_images = 200
for i in range(num_images):
    img_copy = img.copy()

    # 기본 배경 + 랜덤 도형 전체 배치
    rand_bg = generate_random_background(w_rect, h_rect)
    rand_bg = draw_random_shapes(rand_bg, count=200)

    # 책등/가장자리에 도형 집중 배치할 영역 정의
    margin = 200
    region_defs = [
        # 4 모서리
        (0, 0, margin, margin),  # 좌상단
        (w_rect - margin, 0, margin, margin),  # 우상단
        (0, h_rect - margin, margin, margin),  # 좌하단
        (w_rect - margin, h_rect - margin, margin, margin),  # 우하단

        # 위/아래 중앙
        (w_rect // 2 - margin // 2, 0, margin, margin),  # 상단 중앙
        (w_rect // 2 - margin // 2, h_rect - margin, margin, margin),  # 하단 중앙

        # 좌/우 세로 중앙 (책등 강조)
        (0, h_rect // 2 - margin // 2, margin, margin),  # 왼쪽 세로 중앙
        (w_rect - margin, h_rect // 2 - margin // 2, margin, margin),  # 오른쪽 세로 중앙
    ]

    # 각 영역에 도형 n개씩 추가
    for region in region_defs:
        rand_bg = draw_random_shapes(rand_bg, count=15, region=region)

    # 재단선 영역에 최종 이미지 삽입
    img_copy[y:y + h_rect, x:x + w_rect] = rand_bg

    # 저장
    save_path = os.path.join(output_dir, f"3_{i}.png")
    cv2.imwrite(save_path, img_copy)

print(f"{num_images}개의 이미지가 생성되었습니다")


200개의 이미지가 생성되었습니다.
